In [1]:
# import libraries
import pandas as pd
import plotly.graph_objects as go

In [2]:
# load data
companies = pd.read_csv("../data/european_producers_locations.csv", sep=";")
species = pd.read_csv("../data/european_producers_species.csv", sep=";")

In [3]:
# preview companies
companies.head()

,ID_Site,Owner_name,Country,Lat,Long,Pos_info
0,AT01,BDI-BioLife Science GmbH,Austria,"47,258501","15,970157",Confirmed
1,AT02,Ecoduna,Austria,"48,034771","16,815236",Confirmed
2,BE01,Proviron,Belgium,"51,13375","4,341306",Confirmed
3,BE02,Tomalgae,Belgium,"51,032178","3,540922",Confirmed
4,CZ01,EcoFuel Labs,Czech Republic,"50,104569","14,489265",Confirmed


In [4]:
# preview species
species.head()

,ID_Site,ID_Prod_Meth,Species,Unnamed: 3,Unnamed: 4
0,AT01,AT01_Microalgae_Photobioreactors,Haematococcus pluvialis,NaN,NaN
1,AT02,AT02_Microalgae_Photobioreactors,Chlorella sp.,NaN,NaN
2,BE01,BE01_Microalgae_Photobioreactors,Chaetoceros muelleri,NaN,NaN
3,BE01,BE01_Microalgae_Photobioreactors,Diacronema lutheri,NaN,NaN
4,BE01,BE01_Microalgae_Photobioreactors,Nannochloropsis sp.,NaN,NaN


In [5]:
# split data on production and method
species["production"] = [method.split("_")[1] for method in species["ID_Prod_Meth"]]
species["method"] = [method.split("_")[2] for method in species["ID_Prod_Meth"]]

In [6]:
# select macroalgae producers
good_ids = [row["ID_Site"] for _, row in species.iterrows() if row["production"] == "Macroalgae"]
mask_companies = companies["ID_Site"].isin(good_ids)
companies = companies.loc[mask_companies, :].reset_index(drop=True)
mask_species = species["ID_Site"].isin(good_ids)
species = species.loc[mask_species, :].reset_index(drop=True)

In [7]:
# add method info to companies
companies["method"] = ""
for i in range(companies.shape[0]):
    mask_id = species["ID_Site"] == companies.loc[i, "ID_Site"]
    method_current = species.loc[mask_id, "method"].unique()
    if len(method_current) == 1:
        companies.loc[i, "method"] = method_current
    else:
        print(i)
companies["method"] = companies["method"].astype("str")

In [8]:
# discriminate between harvesting and aquaculture and translate in French
companies["method_cat"] = ""
for i in range(companies.shape[0]):
    method_current = companies.loc[i, "method"]
    if ("Harvesting" in method_current) & ("Aquaculture" in method_current):
        companies.loc[i, "method_cat"] = "Récolte & Aquaculture"
    elif "Harvesting" in method_current:
        companies.loc[i, "method_cat"] = "Récolte"
    else:
        companies.loc[i, "method_cat"] = "Aquaculture"

In [9]:
# format numbers from strings to float
columns = ["Lat", "Long"]
for column in columns:
    companies.loc[:, column] = [strnb.replace(",", ".") for strnb in companies.loc[:, column]]
companies[columns] = companies[columns].astype("float")

In [22]:
# plot map of producers
fig=go.Figure()

methods = companies["method_cat"].unique()
colors=["#C63074", "#1C9390", "#F9A115"]
for i in range(len(methods)):
    mask_method = companies["method_cat"] == methods[i]
    fig.add_trace(
        go.Scattermap(
            lon = companies.loc[mask_method, "Long"],
            lat = companies.loc[mask_method, "Lat"],
            mode='markers',
            name=methods[i],
            marker=dict(
                size=10,
                color=colors[i],
            ),
            hovertemplate=companies.loc[mask_method, "Owner_name"] + "<extra></extra>"
        )
    )

fig.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)",
    hovermode="closest",
    hoverlabel=dict(
        font_size=14,
        font_family="Montserrat",
        font_color="#FDF2EE",
        bordercolor="#FDF2EE",
    ),
    hoverdistance=100,
    map=dict(
        bearing=0,
        center=dict(
            lat=54,
            lon=2
        ),
        zoom=2,
        style="carto-positron"
    ),
    modebar=dict(
        orientation="v",
        bgcolor="rgba(0,0,0,0)",
    ),
    legend=dict(
        orientation="h",
        font_family="Montserrat",
        font_size=13,
        font_color="#113972",
        bgcolor="rgba(0,0,0,0)",
        entrywidth=180,
        yanchor="top",
        y=0.0,
        xanchor="center",
        x=0.5
    ),
    margin=dict(
        l=30,
        r=30,
        t=0,
        b=20
    ),
    height=500,
    width=750
)

fig.show()

In [23]:
fig.write_html("../figures/european_producers.html", include_plotlyjs="cdn")